In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd

In [3]:
#import dataset
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [9]:
df.drop(columns=['id','Unnamed: 32'],inplace=True)

In [10]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [18]:
x_train,x_test,y_train,y_test = train_test_split(df.iloc[:,1:],df.iloc[:,0],test_size=0.2,random_state=43)

In [19]:
stander = StandardScaler()
x_train = stander.fit_transform(x_train)
x_test = stander.transform(x_test)

In [20]:
labelEnd = LabelEncoder()
y_train = labelEnd.fit_transform(y_train)
y_test = labelEnd.transform(y_test)

In [21]:
x_train_tensor = torch.tensor(x_train.astype(np.float32))
x_test_tensor = torch.tensor(x_test.astype(np.float32))
y_train_tensor = torch.tensor(y_train.astype(np.float32))
y_test_tensor = torch.tensor(y_test.astype(np.float32))

In [50]:
class CustomDataset(Dataset):
    def __init__(self, features,labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)
    
    def __getitem__(self,index):
        return self.features[index],self.labels[index]

In [52]:
train_dataset = CustomDataset(x_train_tensor,y_train_tensor)
test_dataset = CustomDataset(x_test_tensor,y_test_tensor)

In [54]:
train_dataset.__len__()

455

In [56]:
test_dataset.__getitem__(5)

(tensor([ 0.6050, -0.8085,  0.5114,  0.4530, -0.1445, -0.7005, -0.4270, -0.0951,
         -0.3778, -0.9035, -0.6961, -1.3737, -0.8274, -0.4234,  0.1323, -0.7630,
         -0.2662,  0.4650, -0.8435, -0.5095,  0.2008, -1.2628,  0.0611,  0.0651,
          0.1309, -0.7529, -0.3863,  0.2962, -0.9809, -0.6513]),
 tensor(0.))

In [57]:
class myAnn(nn.Module):
    def __init__(self,n_number):
        super().__init__()
        self.linear = nn.Sequential(
            nn.Linear(n_number,1),
            nn.Sigmoid()
        )
    
    def forward(self,features):
        self.out = self.linear(features)
        return self.out

In [58]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_dataset = DataLoader(test_dataset,batch_size=32,shuffle=True)

In [73]:
for i,j in train_loader:
    print(i)
    print(j)

tensor([[-9.6065e-01, -9.7480e-01, -9.7714e-01, -8.4959e-01, -1.2954e+00,
         -9.9140e-01, -8.3504e-01, -1.0504e+00, -1.4910e+00, -8.5863e-01,
         -5.3798e-01,  7.4340e-02, -5.1261e-01, -4.8376e-01, -1.4804e-01,
         -4.3387e-01, -4.5653e-01, -1.0339e+00, -4.4248e-01, -6.0222e-01,
         -8.9737e-01, -7.9068e-01, -8.8148e-01, -7.7148e-01, -1.0865e+00,
         -8.3775e-01, -8.5278e-01, -1.2184e+00, -9.8091e-01, -9.0360e-01],
        [-4.8953e-01, -9.5885e-01, -5.5436e-01, -5.0971e-01, -1.2017e+00,
         -1.3142e+00, -9.8428e-01, -9.6930e-01, -9.6647e-01, -7.6319e-01,
         -1.0117e+00, -9.8945e-01, -1.0129e+00, -6.6358e-01, -1.2343e+00,
         -1.1673e+00, -1.0328e+00, -1.3527e+00, -3.8239e-01, -1.0459e+00,
         -6.4175e-01, -8.4650e-01, -7.0303e-01, -5.9591e-01, -1.2572e+00,
         -1.1116e+00, -1.0214e+00, -1.1231e+00, -1.6082e-02, -8.9594e-01],
        [-1.1243e+00, -9.9531e-01, -1.1287e+00, -9.7129e-01,  1.2029e+00,
         -4.5110e-01, -9.8042e-01, -

In [59]:
learing_rate =0.1
epochs =45

In [68]:
#create a model
model = myAnn(x_train.shape[1])
# create optimizer
optimizer = torch.optim.SGD(model.parameters(),lr=learing_rate)

In [69]:
loss_function = nn.BCELoss()

In [85]:
#traing loop
for i in range(epochs):
    for batch_features,batch_labels in train_dataset:
        #predict the result
        y_pred = model(batch_features)

        #calculate losss
        loss = loss_function(y_pred.view(-1,1),batch_labels.view(-1,1))

        #clear grad
        optimizer.zero_grad()

        #backpropogation

        loss.backward()

        #update the perameters

        optimizer.step()

    print(f"epochs : {i+1} , loss:  {loss.item()}" )
    print("-"*15)


epochs : 1 , loss:  0.0003685931151267141
---------------
epochs : 2 , loss:  6.485779886133969e-05
---------------
epochs : 3 , loss:  2.06419535970781e-05
---------------
epochs : 4 , loss:  9.069810403161682e-06
---------------
epochs : 5 , loss:  4.701651505456539e-06
---------------
epochs : 6 , loss:  2.7787750696006697e-06
---------------
epochs : 7 , loss:  1.8031197441814584e-06
---------------
epochs : 8 , loss:  1.2475017001634114e-06
---------------
epochs : 9 , loss:  9.04080877717206e-07
---------------
epochs : 10 , loss:  6.792810722799913e-07
---------------
epochs : 11 , loss:  5.252568371361122e-07
---------------
epochs : 12 , loss:  4.157521971137612e-07
---------------
epochs : 13 , loss:  3.3552188938301697e-07
---------------
epochs : 14 , loss:  2.752418311047222e-07
---------------
epochs : 15 , loss:  2.2897064866356232e-07
---------------
epochs : 16 , loss:  1.9279295315755007e-07
---------------
epochs : 17 , loss:  1.6404874259023927e-07
---------------
e